In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Group Project: Data Extraction from BFRSS ASCII file

In [6]:
!pip install gdown
!pip install re
!pip install bs4


ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re


In [ ]:
# Built-in Python library for regular expressions — used to search/extract text patterns (e.g., column numbers, variable names) from the codebook HTML
import re                    

# BeautifulSoup from the 'beautifulsoup4' package — used to parse and navigate HTML/XML documents
from bs4 import BeautifulSoup  

import gdown, io
import pandas as pd

## 1 Parsing the HTML Codebook for ASCII column numbers

In [ ]:
# Shared Google Drive file ID for the codebook HTML file
FILE_ID = "1ktq0TuFxtKNtWLUBGnauNYJAgQTUmkv1"
url = f"https://drive.google.com/uc?id={FILE_ID}"


In [23]:
# --- Parse HTML codebook ---
# Download the HTML codebook from Google Drive to a local file
content = gdown.download(url, quiet=False)

# Open the downloaded file and parse it with BeautifulSoup
with open(content, encoding="windows-1252") as f:
    soup = BeautifulSoup(f, "html.parser")


Downloading...
From: https://drive.google.com/uc?id=1ktq0TuFxtKNtWLUBGnauNYJAgQTUmkv1
To: d:\Dropbox (Personal)\2_TEACHING_FOLDER\_SMU_Courses\ISSS623_MITB_HealthcareAnalytics\Lectures\2026\Lesson 1\Github_repo\ISSS623_2024\Lecture 1\USCODE24_LLCP_082125.HTML
100%|██████████| 1.61M/1.61M [00:00<00:00, 37.1MB/s]


In [24]:
col_pattern   = re.compile(r"Column:\s*(\d+)(?:-(\d+))?")
name_pattern  = re.compile(r"SAS\s+Variable\s+Name:\s*(\S+)")
type_pattern  = re.compile(r"Type\s+of\s+Variable:\s*(Num|Char)", re.IGNORECASE)
label_pattern = re.compile(r"Label:\s*(.+?)(?:\n|Section\s+Name:)", re.DOTALL)

records = []
seen = set()

for td in soup.find_all("td", class_="linecontent"):
    text = td.get_text(" ").replace('\xa0', ' ')
    col_m   = col_pattern.search(text)
    name_m  = name_pattern.search(text)
    type_m  = type_pattern.search(text)
    label_m = label_pattern.search(text)
    if not (col_m and name_m):
        continue
    varname = name_m.group(1)
    if varname in seen:
        continue
    seen.add(varname)
    start   = int(col_m.group(1))
    end     = int(col_m.group(2)) if col_m.group(2) else start
    dtype   = "str" if (type_m and type_m.group(1).lower() == "char") else "num"
    label   = label_m.group(1).strip() if label_m else ""
    records.append({
        "variable": varname,
        "label":    label,
        "col_start": start,
        "col_end":   end,
        "type":      dtype,
    })

vardict = pd.DataFrame(records)
print(f"Variables parsed: {len(records)}")
print("First 5:", records[:5])

Variables parsed: 297
First 5: [{'variable': '_STATE', 'label': 'State FIPS Code', 'col_start': 1, 'col_end': 2, 'type': 'num'}, {'variable': 'FMONTH', 'label': 'File Month', 'col_start': 17, 'col_end': 18, 'type': 'num'}, {'variable': 'IDATE', 'label': 'Interview Date', 'col_start': 19, 'col_end': 26, 'type': 'str'}, {'variable': 'IMONTH', 'label': 'Interview Month', 'col_start': 19, 'col_end': 20, 'type': 'str'}, {'variable': 'IDAY', 'label': 'Interview Day', 'col_start': 21, 'col_end': 22, 'type': 'str'}]


# 2. Sample Data Extraction Codes for Potential Topics

## 2.1 Topic A: Health Burden

Example question: Can BRFSS respondent characteristics predict high health burden?


In [18]:
# Map specific variables to their Topic A categories; all others will be NaN
topic_a_fields = {
    "GENHLTH":  "Outcome components",
    "PHYSHLTH": "Outcome components",
    "MENTHLTH": "Outcome components",
    "POORHLTH": "Outcome components",
    "_AGE80":   "Demographics",
    "SEXVAR":   "Demographics",
    "_RACEGR3": "Demographics",
    "MARITAL":  "Demographics",
    "_EDUCAG":  "Demographics",
    "_INCOMG1": "Demographics",
    "EMPLOY1":  "Demographics",
    "_URBSTAT": "Demographics",
    "_TOTINDA": "Health behaviours",
    "_SMOKER3": "Health behaviours",
    "_RFBING6": "Health behaviours",
    "_BMI5CAT": "Health behaviours",
    "_HLTHPL2": "Healthcare access",
    "PERSDOC3": "Healthcare access",
    "MEDCOST1": "Healthcare access",
    "CHECKUP1": "Healthcare access",
    "_MICHD":   "Chronic conditions",
    "CVDSTRK3": "Chronic conditions",
    "_CASTHM1": "Chronic conditions",
    "CHCCOPD3": "Chronic conditions",
    "ADDEPEV3": "Chronic conditions",
    "CHCKDNY2": "Chronic conditions",
    "_DRDXAR2": "Chronic conditions",
    "DIABETE4": "Chronic conditions",
    "_STATE":   "Geographical Filter",
}

vardict["Topic A Possible Fields"] = vardict["variable"].map(topic_a_fields)




## 2.2 Topic B: Mental Health

Example question: Can demographic, behavioural, healthcare access, chronic disease, and disability variables predict mental health issues among BRFSS respondents?


In [25]:
topic_b_fields = {
    "MENTHLTH": "Outcome",
    "_AGEG5YR": "Demographics",
    "SEXVAR":   "Demographics",
    "_RACEGR3": "Demographics",
    "MARITAL":  "Demographics",
    "_EDUCAG":  "Demographics",
    "_INCOMG1": "Demographics",
    "EMPLOY1":  "Demographics",
    "_URBSTAT": "Demographics",
    "_TOTINDA": "Behaviours",
    "_SMOKER3": "Behaviours",
    "_CURECI3": "Behaviours",
    "_RFBING6": "Behaviours",
    "_RFDRHV9": "Behaviours",
    "_BMI5CAT": "Behaviours",
    "_HLTHPL2": "Healthcare access",
    "PERSDOC3": "Healthcare access",
    "MEDCOST1": "Healthcare access",
    "CHECKUP1": "Healthcare access",
    "_MICHD":   "Chronic diseases",
    "CVDSTRK3": "Chronic diseases",
    "_CASTHM1": "Chronic diseases",
    "CHCCOPD3": "Chronic diseases",
    "CHCKDNY2": "Chronic diseases",
    "_DRDXAR2": "Chronic diseases",
    "DIABETE4": "Chronic diseases",
    "DEAF":     "Disabilities",
    "BLIND":    "Disabilities",
    "DECIDE":   "Disabilities",
    "DIFFWALK": "Disabilities",
    "DIFFDRES": "Disabilities",
    "DIFFALON": "Disabilities",
    "_STATE":   "Geographical Filter",
}

vardict["Topic B Possible Fields"] = vardict["variable"].map(topic_b_fields)

## 2.3 Topic C: Preventive Health

Example question: Can BRFSS respondent characteristics predict whether a respondent has a preventive care gap?


In [26]:
topic_c_fields = {
    "CHECKUP1": "Outcome",
    "_AGEG5YR": "Demographics",
    "SEXVAR":   "Demographics",
    "_RACEGR3": "Demographics",
    "MARITAL":  "Demographics",
    "_EDUCAG":  "Demographics",
    "_INCOMG1": "Demographics",
    "EMPLOY1":  "Demographics",
    "_URBSTAT": "Demographics",
    "_HLTHPL2": "Healthcare access",
    "PRIMINS2": "Healthcare access",
    "PERSDOC3": "Healthcare access",
    "MEDCOST1": "Healthcare access",
    "_DENVST3": "Healthcare access",
    "_TOTINDA": "Behaviours",
    "_SMOKER3": "Behaviours",
    "_CURECI3": "Behaviours",
    "_RFBING6": "Behaviours",
    "_BMI5CAT": "Behaviours",
    "GENHLTH":  "Health and chronic conditions",
    "PHYSHLTH": "Health and chronic conditions",
    "MENTHLTH": "Health and chronic conditions",
    "_MICHD":   "Health and chronic conditions",
    "CVDSTRK3": "Health and chronic conditions",
    "_CASTHM1": "Health and chronic conditions",
    "CHCCOPD3": "Health and chronic conditions",
    "ADDEPEV3": "Health and chronic conditions",
    "CHCKDNY2": "Health and chronic conditions",
    "_DRDXAR2": "Health and chronic conditions",
    "DIABETE4": "Health and chronic conditions",
    "DEAF":     "Disabilities",
    "BLIND":    "Disabilities",
    "DECIDE":   "Disabilities",
    "DIFFWALK": "Disabilities",
    "DIFFDRES": "Disabilities",
    "DIFFALON": "Disabilities",
    "_STATE":   "Geographical Filter",
}

vardict["Topic C Possible Fields"] = vardict["variable"].map(topic_c_fields)

In [27]:

vardict.to_csv("brfss2024_variable_dictionary.csv", index=False)

print(f"Saved {len(vardict)} variables to CSV.")
vardict.head(10)

Saved 297 variables to CSV.


,variable,label,col_start,col_end,type,Topic B Possible Fields,Topic C Possible Fields
0,_STATE,State FIPS Code,1,2,num,Geographical Filter,Geographical Filter
1,FMONTH,File Month,17,18,num,NaN,NaN
2,IDATE,Interview Date,19,26,str,NaN,NaN
3,IMONTH,Interview Month,19,20,str,NaN,NaN
4,IDAY,Interview Day,21,22,str,NaN,NaN
5,IYEAR,Interview Year,23,26,str,NaN,NaN
6,DISPCODE,Final Disposition,32,35,num,NaN,NaN
7,SEQNO,Annual Sequence Number,36,45,str,NaN,NaN
8,_PSU,Primary Sampling Unit,36,45,num,NaN,NaN
9,CTELENM1,Correct telephone number?,63,63,num,NaN,NaN


### Moving saved files to Google Drive

To move the generated CSV files to your Google Drive, you can use the `mv` command. Replace with the actual folder name in your Google Drive where you want to save the files.

In [ ]:
# Move the CSV file to Google Drive
import os

drive_path = '/content/drive/My Drive/ISSS623/Lecture_1_Outputs' 

# Create the folder if it doesn't exist
os.makedirs(drive_path, exist_ok=True)

# Move the files
!mv brfss2024_variable_dictionary.csv "{drive_path}"

print(f"Files moved to {drive_path}")

# 3 Extracting data from BRFSS based on selected variables (Note: there are 400K+ records in BRFSS 2024)

In [ ]:
ASC_PATH      = r"LLCP2024.ASC"

## 3.1 Extract Topic A candidate variables

In [ ]:
# Filter vardict to only Topic A variables using the already-defined topic_a_fields dictionary
vardict_a = vardict[vardict["variable"].isin(topic_a_fields.keys())].copy()

# Build colspecs, names, and dtype_map from Topic A variables only
colspecs  = [(row["col_start"] - 1, row["col_end"]) for _, row in vardict_a.iterrows()]
names     = vardict_a["variable"].tolist()
dtype_map = {row["variable"]: str for _, row in vardict_a.iterrows() if row["type"] == "str"}

# Read only Topic A columns from the fixed-width ASCII file
df_a = pd.read_fwf(
    ASC_PATH,
    colspecs=colspecs,
    names=names,
    dtype=dtype_map,
    encoding="windows-1252"
    #,
    #nrows=50,   # remove for full load
)

print(df_a.shape)
df_a.head()

(457670, 29)


,_STATE,SEXVAR,GENHLTH,PHYSHLTH,MENTHLTH,POORHLTH,PERSDOC3,MEDCOST1,CHECKUP1,CVDSTRK3,...,_MICHD,_CASTHM1,_DRDXAR2,_RACEGR3,_AGE80,_BMI5CAT,_EDUCAG,_INCOMG1,_SMOKER3,_RFBING6
0,1,2,3.0,2.0,88.0,88.0,2.0,2.0,1.0,2.0,...,2.0,1,1.0,1,78,2.0,2,9,4,1
1,1,1,1.0,88.0,88.0,NaN,1.0,2.0,1.0,2.0,...,1.0,1,1.0,1,80,3.0,4,7,3,1
2,1,1,2.0,30.0,88.0,1.0,3.0,1.0,4.0,2.0,...,2.0,1,1.0,1,59,2.0,3,9,1,2
3,1,1,1.0,88.0,88.0,NaN,1.0,2.0,1.0,2.0,...,2.0,1,1.0,1,80,3.0,4,4,4,1
4,1,1,3.0,88.0,88.0,NaN,1.0,2.0,1.0,2.0,...,2.0,1,2.0,1,47,2.0,3,2,4,1


## 3.2 Extract Topic B candidate variables

In [31]:
# Filter vardict to only Topic B variables using the already-defined topic_b_fields dictionary
vardict_b = vardict[vardict["variable"].isin(topic_b_fields.keys())].copy()

# Build colspecs, names, and dtype_map from Topic B variables only
colspecs  = [(row["col_start"] - 1, row["col_end"]) for _, row in vardict_b.iterrows()]
names     = vardict_b["variable"].tolist()
dtype_map = {row["variable"]: str for _, row in vardict_b.iterrows() if row["type"] == "str"}

# Read only Topic B columns from the fixed-width ASCII file
df_b = pd.read_fwf(
    ASC_PATH,
    colspecs=colspecs,
    names=names,
    dtype=dtype_map,
    encoding="windows-1252"
    #,
    #nrows=50,   # remove for full load
)

print(df_b.shape)
df_b.head()

(457670, 33)


,_STATE,SEXVAR,MENTHLTH,PERSDOC3,MEDCOST1,CHECKUP1,CVDSTRK3,CHCCOPD3,CHCKDNY2,DIABETE4,...,_DRDXAR2,_RACEGR3,_AGEG5YR,_BMI5CAT,_EDUCAG,_INCOMG1,_SMOKER3,_CURECI3,_RFBING6,_RFDRHV9
0,1,2,88.0,2.0,2.0,1.0,2.0,2.0,2.0,3.0,...,1.0,1,12,2.0,2,9,4,1,1,1
1,1,1,88.0,1.0,2.0,1.0,2.0,2.0,2.0,3.0,...,1.0,1,13,3.0,4,7,3,1,1,1
2,1,1,88.0,3.0,1.0,4.0,2.0,2.0,2.0,3.0,...,1.0,1,8,2.0,3,9,1,1,2,1
3,1,1,88.0,1.0,2.0,1.0,2.0,2.0,2.0,3.0,...,1.0,1,13,3.0,4,4,4,1,1,1
4,1,1,88.0,1.0,2.0,1.0,2.0,2.0,2.0,3.0,...,2.0,1,6,2.0,3,2,4,1,1,1


## 3.3 Extract Topic C candidate variables

In [33]:
# Filter vardict to only Topic C variables using the already-defined topic_c_fields dictionary
vardict_c = vardict[vardict["variable"].isin(topic_c_fields.keys())].copy()

# Build colspecs, names, and dtype_map from Topic C variables only
colspecs  = [(row["col_start"] - 1, row["col_end"]) for _, row in vardict_c.iterrows()]
names     = vardict_c["variable"].tolist()
dtype_map = {row["variable"]: str for _, row in vardict_c.iterrows() if row["type"] == "str"}

# Read only Topic C columns from the fixed-width ASCII file
df_c = pd.read_fwf(
    ASC_PATH,
    colspecs=colspecs,
    names=names,
    dtype=dtype_map,
    encoding="windows-1252"
    #,
    #nrows=50,   # remove for full load
)

print(df_c.shape)
df_c.head()

(457670, 37)


,_STATE,SEXVAR,GENHLTH,PHYSHLTH,MENTHLTH,PRIMINS2,PERSDOC3,MEDCOST1,CHECKUP1,CVDSTRK3,...,_CASTHM1,_DRDXAR2,_RACEGR3,_AGEG5YR,_BMI5CAT,_EDUCAG,_INCOMG1,_SMOKER3,_CURECI3,_RFBING6
0,1,2,3.0,2.0,88.0,3.0,2.0,2.0,1.0,2.0,...,1,1.0,1,12,2.0,2,9,4,1,1
1,1,1,1.0,88.0,88.0,3.0,1.0,2.0,1.0,2.0,...,1,1.0,1,13,3.0,4,7,3,1,1
2,1,1,2.0,30.0,88.0,1.0,3.0,1.0,4.0,2.0,...,1,1.0,1,8,2.0,3,9,1,1,2
3,1,1,1.0,88.0,88.0,3.0,1.0,2.0,1.0,2.0,...,1,1.0,1,13,3.0,4,4,4,1,1
4,1,1,3.0,88.0,88.0,5.0,1.0,2.0,1.0,2.0,...,1,2.0,1,6,2.0,3,2,4,1,1


In [34]:
# Save each topic's DataFrame to a CSV file in the current folder
df_a.to_csv("brfss2024_topic_a.csv", index=False)
df_b.to_csv("brfss2024_topic_b.csv", index=False)
df_c.to_csv("brfss2024_topic_c.csv", index=False)


# Final Remarks:
### Final remarks and cautions

1. **Define one primary outcome clearly**

   * Topic A: poor/fair health, frequent physical distress, frequent mental distress, activity limitation, or a justified composite.
   * Topic B: frequent mental distress for example `MENTHLTH ≥ 14 days`.
   * Topic C: select one preventive-care gap, such as no annual check-up, cost barrier, no dental visit, or no flu vaccination.

2. **Avoid target leakage**

   * Do not include the outcome variable, its derived version, or variables directly used to construct a composite as predictors.
   * Examples: exclude `_MENT14D` when predicting an outcome derived from `MENTHLTH`; exclude `_DENVST3` when predicting dental-care gaps.

3. **Use one version of each construct**

   * Avoid including both raw and derived forms, such as `EDUCA` and `_EDUCAG`, or `SEXVAR` and `_SEX`.

4. **Check BRFSS coding carefully**

   * Recode “don’t know,” “refused,” “not asked,” and other special values as missing before analysis.
   * Do not treat structural missingness as absence of the condition or behaviour.

5. **Check optional-module coverage**

   * Some behavioural, psychosocial, marijuana-use, and social-determinant variables were administered only in selected states or questionnaire versions.
   * Try not to use these specialized data. If needed, always check and report the jurisdictions and eligible population included.

6. **Assess missing data and class imbalance**

   * Report missingness by variable and outcome.
   * Use appropriate validation metrics when the positive outcome is uncommon, including sensitivity, specificity, F1 score, precision–recall AUC, and ROC AUC.

7. **Prevent overfitting**

   * Limit the initial predictor set, use cross-validation, and evaluate performance on a separate test set.
   * Avoid selecting variables using the full dataset before splitting.

8. **Interpret results within the BRFSS context**

    * BRFSS data are self-reported, cross-sectional, and subject to recall, reporting, nonresponse, and coverage bias.
    * Findings should be described as applying to the analytic BRFSS respondent population, not necessarily to all individuals. 

9. **Check variable definitions and completeness**

    * Review the official BRFSS codebook and variable layout to confirm each variable’s definition, response categories, eligible population, and reference period.
    * Examine variable completeness and exclude or justify fields with substantial missingness or limited coverage.

10. **Begin with an explainable regression model**

    * Fit an interpretable regression model, such as logistic regression, as the initial baseline before applying more advanced machine-learning models.
    * Compare advanced models against the baseline in terms of predictive performance, interpretability, calibration, and practical usefulness.

